In [ ]:
# 安装 uv，并创建指定 Python 版本的虚拟环境（适用于 Colab/Linux）。
# 在当前 Notebook 内核环境中安装固定版本的 uv；-q 减少安装日志。
%pip install -q "uv==0.13.6"
# 查看 uv 版本，并下载由 uv 管理的 Python 3.12.13。
!uv --version
!uv python install 3.12.13
# 仅在当前目录没有 .venv 时创建；已有目录会直接复用，不检查其中的 Python 版本。
!if [ ! -d .venv ]; then uv venv --managed-python --python 3.12.13 .venv; else echo ".venv 已存在，跳过创建"; fi

uv 0.13.6 (x86_64-unknown-linux-gnu)
Installed Python 3.12.13 in 1.17s
 + cpython-3.12.13-linux-x86_64-gnu (python3.12)
Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [ ]:
%%script .venv/bin/python

# 通过上面的单元格魔法，在虚拟环境的 Python 子进程中检查解释器信息。
# %%script 必须位于首行；它只影响本单元格，不会切换 Notebook 内核。

import sys
import platform

# 版本和可执行文件路径用于确认实际调用的是哪个 Python。
print("Python 完整版本：", sys.version)
print("Python 路径：", sys.executable)
# 虚拟环境中，sys.prefix 指向环境目录，sys.base_prefix 指向基础解释器目录。
print("虚拟环境目录：", sys.prefix)
print("基础 Python 目录：", sys.base_prefix)
# 输出 CPU 架构，例如 x86_64。
print("系统架构：", platform.machine())

Python 完整版本： 3.12.13 (main, Aug  7 2026, 02:25:39) [Clang 22.1.3 ]
Python 路径： /content/.venv/bin/python
虚拟环境目录： /content/.venv
基础 Python 目录： /root/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu
系统架构： x86_64


In [ ]:
# 检查 GPU、FFmpeg，并向 /content/.venv 安装固定版本的依赖。
# 后续路径按 Colab 的 /content 工作目录编写，需与单元格 1 创建的环境一致。
# 查看显卡、驱动和显存状态；此处显示的 CUDA 版本表示驱动支持的版本。
!nvidia-smi
# 安装 FFmpeg，供后续音视频处理使用。
!apt-get install -y ffmpeg
# 只打印 FFmpeg 版本信息的第一行。
!ffmpeg -version | head -n 1
# --python 指定安装目标；--torch-backend cu128 选择 CUDA 12.8 对应的后端。
# torch 用于张量计算，torchvision 提供视觉工具，torchcodec 用于音视频解码。
# numpy 用于数组计算；nvidia-npp-cu12 提供后续配置的 NVIDIA NPP 动态库。
# 下面是反斜杠续行的完整命令，注释放在命令前，避免打断续行。
!uv pip install \
    --python /content/.venv/bin/python \
    --torch-backend cu128 \
    "torch==2.11.0+cu128" \
    "torchvision==0.26.0+cu128" \
    "torchcodec==0.11.0+cu128" \
    "numpy==2.0.2" \
    "nvidia-npp-cu12==12.4.1.87"

Fri Oct 30 07:31:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 在当前 Notebook 内核中配置 NPP 动态库路径，供后续子进程继承。
import os
from pathlib import Path

# 指向虚拟环境内 NPP 的库目录；该路径与 /content/.venv 和 Python 3.12 对应。
npp_dir = Path(
    "/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib"
)

# 检查目标动态库是否存在
libs = list(npp_dir.glob("libnppicc.so*"))
print("NPP 库文件:", libs)

# 未找到目标库时立即报错，提示先检查依赖安装。
assert libs, "NPP 库未找到，请检查安装结果"

# 为后续 %%script 启动的 Python 子进程配置动态库搜索路径
# 保留已有路径；Linux 下多个动态库搜索目录使用冒号分隔。
old_path = os.environ.get("LD_LIBRARY_PATH", "")
paths = old_path.split(":") if old_path else []

# 将 NPP 目录放在搜索路径前面，并避免重复运行时重复添加。
if str(npp_dir) not in paths:
    os.environ["LD_LIBRARY_PATH"] = (
        str(npp_dir) + (":" + old_path if old_path else "")
    )

print("NPP 动态库路径配置成功")

NPP 库文件: [PosixPath('/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib/libnppicc.so.12')]
NPP 动态库路径配置成功


In [ ]:
%%script /content/.venv/bin/python

# 在虚拟环境子进程中检查依赖版本，并执行一次 GPU 矩阵乘法。
# 先运行单元格 4，使此子进程继承配置好的 NPP 动态库搜索路径。
import torch
import torchvision
import numpy as np
import torchcodec

# 打印实际导入的库版本，便于核对安装结果。
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("TorchCodec:", torchcodec.__version__)
print("NumPy:", np.__version__)

# 这里的 CUDA 版本是 PyTorch 构建所用的版本；is_available() 检查 CUDA 是否可用。
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

# 在 GPU 上创建 1024×1024 的随机矩阵并相乘；CUDA 不可用时这里会报错。
x = torch.randn(1024, 1024, device="cuda")
y = x @ x

# 输出结果形状，用于基本冒烟检查；此处没有显式同步 GPU 或校验数值精度。
print("GPU 计算测试通过:", y.shape)

PyTorch: 2.11.0+cu128
torchvision: 0.26.0+cu128
TorchCodec: 0.11.0+cu128
NumPy: 2.0.2
CUDA: 12.8
GPU available: True
GPU 计算测试通过: torch.Size([1024, 1024])
